In [3]:
import sys
from pathlib import Path

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src").is_dir() and (candidate / "project.ipynb").is_file():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pyspark.sql import SparkSession
from src.dataOI import DataOI

SEED = 42

# True: read the raw CSV, then clean, engineer text, split, encode, and save that split.
# False: skip those steps and load the saved train, validation, and test frames from data/processed/split.
REBUILD = True

# `spark.driver.memory` is read only when the JVM starts. Restart the kernel after changing it.
spark = SparkSession.builder \
                            .master("local[*]")\
                            .appName("DrugReviewClassification")\
                            .config("spark.driver.memory","4g")\
                            .getOrCreate()
spark_context = spark.sparkContext
spark_context.setLogLevel("ERROR")
data_oi = DataOI()


In [4]:
from src.cleaning import data_cleaning
from src.text_features import text_feature_engineering
from src.encoder import encoder
from src.split import split_and_weight

if REBUILD:
    df_raw = data_oi.load_raw_data()
    df_cleaned = data_cleaning(df_raw)
    df_cleaned.cache().count()
    df_engineered = text_feature_engineering(df_cleaned,True)
    df_engineered.cache().count()
    df_train, df_valid, df_test = split_and_weight(df_engineered, SEED)
    df_train, df_valid, df_test = encoder(df_train, df_valid, df_test)
    data_oi.save_split_data(df_train, df_valid, df_test)
else:
    df_train, df_valid, df_test = data_oi.load_split_data()


Data Cleaning
After dropping rows with a null `condition`
Row count: 161297 -> 160398, removed 899 rows
After dropping rows whose `condition` contains '</span> users found this comment helpful'
Row count: 160398 -> 159498, removed 900 rows
Before replacement: 15212 `condition` values match configured repairs, 104180 `review` values contain HTML entities
After replacement: 0 `condition` values still match configured repairs, 0 `review` values contain HTML entities
After keeping the row with the highest `usefulCount` for each `review`
Row count: 159498 -> 111167, removed 48331 rows
Added `usefulCount_log` = log(1 + `usefulCount`)
+---------------+------------------+-----------------+-----------------+
|         column|          variance|           stddev|         skewness|
+---------------+------------------+-----------------+-----------------+
|    usefulCount| 1281.077141590041|35.79213798573705|4.488040210031037|
|usefulCount_log|1.4185266277513175|1.191019155073216|-0.29665492455691|

d:\python\envs\DataEnv\Lib\site-packages\pyspark\sql\udf.py:120: RuntimeWarning: Arrow optimization failed to enable because PyArrow or Pandas is not installed. Falling back to a non-Arrow-optimized UDF.
  warnings.warn(


In [5]:
from src.embeddings import EmbeddingModel
# Fit the selected embedding before prediction; downstream models read `embedding`.
embedding_model = EmbeddingModel(df_train, df_valid, df_test)
embedding_name = "tfidf"

print(f"Embedding Model `{embedding_name}` Training")
# EmbeddingModel.train_registry[embedding_name](embedding_model)


Embedding Model `tfidf` Training


In [10]:
train_embedding_dict = {}
valid_embedding_dict = {}
test_embedding_dict = {}

print(f"Embedding Model `{embedding_name}` Running")
train_embedding_dict[embedding_name], valid_embedding_dict[embedding_name], test_embedding_dict[embedding_name] = EmbeddingModel.predict_registry[embedding_name](embedding_model)
data_oi.save_embedding_data(
    embedding_name,
    train_embedding_dict[embedding_name],
    valid_embedding_dict[embedding_name],
    test_embedding_dict[embedding_name],
)


Embedding Model `tfidf` Running


In [ ]:
from src.analytics import AnalyticsModel

analytics_models = {}
analytics_models[embedding_name] = AnalyticsModel(
    embedding_name,
    train_embedding_dict[embedding_name],
    valid_embedding_dict[embedding_name],
    test_embedding_dict[embedding_name],
)


In [ ]:
train_pred_dict = {}
valid_pred_dict = {}
test_pred_dict = {}
for name, function in AnalyticsModel.registry.items():
    print(f"Embedding Model `{embedding_name}` Analytics Model `{name}` Running")
    # Format: `{analytics}__{embedding}`. Either name may contain a single underscore.
    model_name = f"{name}__{embedding_name}"
    train_pred_dict[model_name], valid_pred_dict[model_name], test_pred_dict[model_name] = function(
        analytics_models[embedding_name]
    )
    data_oi.save_prediction_data(
        model_name,
        train_pred_dict[model_name],
        valid_pred_dict[model_name],
        test_pred_dict[model_name],
    )


In [ ]:
from src.evaluation import Scorer

# Score only predictions produced by this experiment.
scorer_train = Scorer(train_pred_dict)
print("Training Data:")
scorer_train.show_metrics(truncate = 80)

scorer_valid = Scorer(valid_pred_dict)
print("Validation Data:")
scorer_valid.show_metrics(truncate = 80)

scorer_test = Scorer(test_pred_dict)
print("Testing Data:")
scorer_test.show_metrics(truncate = 80)